# L08 capstone: your autograd engine against PyTorch

This notebook needs no GPU; any Colab runtime works. Run the cells in order. The last cell is the
autograder: it runs the capstone tests and prints a completion code to paste into the lesson.

You will (1) paste your engine, (2) train a 2-64-3 ReLU network on the spiral with your engine and
with PyTorch from identical initial weights and compare the loss curves, (3) walk a real PyTorch
`grad_fn` graph, and (4) write a custom `torch.autograd.Function` that passes `gradcheck`.

In [ ]:
import os
import numpy as np
import torch

SMALL = os.environ.get("SMALL") == "1"
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print("PyTorch", torch.__version__, "(SMALL mode)" if SMALL else "")

## 1. Your engine

Paste your finished engine from exercise 9 of the lesson (the `Tensor` class with its operations,
`cross_entropy_fused`, `Parameter`, `Linear`, `MLP` and `SGD`) into the next cell, replacing the
`raise` line.

In [ ]:
raise NotImplementedError("Paste your engine from exercise 9 here")

## 2. Data and the engine run

In [ ]:
def make_spiral(n=100, K=3, seed=0):
    rng = np.random.default_rng(seed)
    Xs, ys = [], []
    for j in range(K):
        r = np.linspace(0.0, 1.0, n)
        t = np.linspace(4 * j, 4 * (j + 1), n) + 0.2 * rng.normal(size=n)
        Xs.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        ys.append(np.full(n, j))
    return np.concatenate(Xs), np.concatenate(ys)


X, y = make_spiral()
SIZES = [2, 64, 3]
STEPS = 40 if SMALL else 300
LR = 1.0

model = MLP(SIZES, seed=0)
init_weights = [(layer.W.data.copy(), layer.b.data.copy()) for layer in model.layers]
opt = SGD(model.parameters(), LR)
engine_losses = []
for _ in range(STEPS):
    opt.zero_grad()
    loss = cross_entropy_fused(model(Tensor(X)), y)
    loss.backward()
    opt.step()
    engine_losses.append(float(loss.data))
print("engine: first loss", round(engine_losses[0], 4), "last loss", round(engine_losses[-1], 4))

## 3. The same network in PyTorch

Build `TorchMLP` from the engine's initial weights. Note that `torch.nn.Linear` stores its weight as
`(out_features, in_features)`, the transpose of your engine's `W`. Then train it with
`torch.optim.SGD` and `torch.nn.functional.cross_entropy` for the same number of steps.

In [ ]:
class TorchMLP(torch.nn.Module):
    def __init__(self, init):
        super().__init__()
        raise NotImplementedError("Write your code here")

    def forward(self, x):
        raise NotImplementedError("Write your code here")


tmodel = TorchMLP(init_weights)
topt = torch.optim.SGD(tmodel.parameters(), lr=LR)
Xt, yt = torch.tensor(X), torch.tensor(y)
torch_losses = []
for _ in range(STEPS):
    topt.zero_grad()
    tloss = torch.nn.functional.cross_entropy(tmodel(Xt), yt)
    tloss.backward()
    topt.step()
    torch_losses.append(tloss.item())
print("largest difference between the loss curves:", max(abs(a - b) for a, b in zip(engine_losses, torch_losses)))

## 4. Walk the graph PyTorch recorded

Every non-leaf tensor has a `grad_fn`; its `next_functions` are `(function, index)` pairs leading to
the functions that produced its inputs. Write `walk_grad_fn(t)` returning the class names of every
function reachable from `t.grad_fn` (leaf nodes appear as `AccumulateGrad`; `None` entries mark
inputs that need no gradient).

In [ ]:
def walk_grad_fn(t):
    raise NotImplementedError("Write your code here")


grad_fn_names = walk_grad_fn(torch.nn.functional.cross_entropy(tmodel(Xt), yt))
print(sorted(set(grad_fn_names)))

## 5. A custom autograd function

Implement the squared ReLU, $f(x) = \max(0, x)^2$, as a `torch.autograd.Function` with your own
backward rule, then verify it with `torch.autograd.gradcheck` and against PyTorch's automatic
derivative of the same expression.

In [ ]:
class SquaredReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        raise NotImplementedError("Write your code here")

    @staticmethod
    def backward(ctx, g):
        raise NotImplementedError("Write your code here")


xc = torch.randn(25, requires_grad=True)
custom_gradcheck_ok = bool(torch.autograd.gradcheck(SquaredReLU.apply, (xc,)))
(g_custom,) = torch.autograd.grad(SquaredReLU.apply(xc).sum(), xc)
(g_auto,) = torch.autograd.grad((torch.relu(xc) ** 2).sum(), xc)
custom_matches_autograd = float((g_custom - g_auto).abs().max())
print("gradcheck:", custom_gradcheck_ok, "| max difference from autograd:", custom_matches_autograd)

In [ ]:
# Forge autograder: run this cell last. It runs the capstone tests and prints your completion code.
import hashlib as _hashlib, sys as _sys, types as _types, unittest as _unittest
_FORGE_ID = "L08"
_FORGE_TESTS = '''import numpy as np
import sub


def test_engine_matches_pytorch():
    e = np.array(sub.engine_losses)
    t = np.array(sub.torch_losses)
    assert len(e) == len(t) >= 20
    assert np.max(np.abs(e - t)) < 1e-6, np.max(np.abs(e - t))


def test_engine_learns():
    assert sub.engine_losses[-1] < 0.7 * sub.engine_losses[0]


def test_grad_fn_walk():
    ops = set(sub.grad_fn_names)
    assert any("Addmm" in o or "Mm" in o for o in ops), ops
    assert any("Relu" in o for o in ops), ops
    assert any("NllLoss" in o or "LogSoftmax" in o for o in ops), ops
    assert "AccumulateGrad" in ops


def test_custom_function():
    assert sub.custom_gradcheck_ok is True
    assert sub.custom_matches_autograd < 1e-12
'''
sub = _types.SimpleNamespace(**{_k: _v for _k, _v in globals().items() if not _k.startswith('_')})
_sys.modules['sub'] = sub
_ns = {'__name__': 'forge_tests'}
exec(compile(_FORGE_TESTS, 'tests.py', 'exec'), _ns)
_passed = []
for _name, _fn in list(_ns.items()):
    if _name.startswith('test_') and callable(_fn):
        try:
            _fn()
            _passed.append(_name)
            print('PASS', _name)
        except _unittest.SkipTest as _e:
            print('SKIP', _name, '-', _e)
        except Exception as _e:
            print('FAIL', _name, '-', type(_e).__name__, _e)
_digest = _hashlib.sha256(f"{_FORGE_ID}:{','.join(sorted(_passed))}:forge".encode()).hexdigest()[:8]
print(f"Completion code: FORGE-{_FORGE_ID}-{_digest}")